# Camargo Sepsis (improved) — SharedCat + ALL Sepsis features + n-gram=5

Identical to `../../../old/Training/notebook/train_camargo_LSTM.ipynb` (SharedCat_LSTM, paper Table 3 hyperparameters, `NGRAM_SIZE=5`, `lr=1e-3`, 100 epochs) **with one change**: the loader pickles encoded under `../Loader/pkl/Sepsis_all_5_allfeat_*.pkl` include every Sepsis CSV column (24 extra clinical-flag categoricals + 4 lab-value numerics + Age) on top of the original `concept:name + org:group + case_elapsed_time`. The trainer's `model_feat` block automatically picks up everything in the dataset, so SharedCat now embeds all categoricals and consumes all numericals.

Reimplementation for comparison:
- Camargo, Manuel, Marlon Dumas, and Oscar González-Rojas. "Learning accurate LSTM models of business processes." BPM 2019.
- https://github.com/AdaptiveBProcess/GenerativeLSTM/

# Imports

In [1]:
import importlib
import sys
import torch
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir() and (_current / 'data').is_dir():
        break
    _current = _current.parent
_project_root = _current

for p in [
    _project_root,
    _project_root / 'src',
    _project_root / 'src' / 'reimplemented_comparable_approaches' / 'camargo_LSTM_suffix_pred',
]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Data

### Load Data Files

In [2]:
file_path_train = '../../Loader/pkl/Sepsis_all_5_allfeat_train.pkl'
sepsis_train_dataset = torch.load(file_path_train, weights_only=False)
print(type(sepsis_train_dataset))

file_path_val = '../../Loader/pkl/Sepsis_all_5_allfeat_val.pkl'
sepsis_val_dataset = torch.load(file_path_val, weights_only=False)
print(type(sepsis_val_dataset))


<class 'event_log_loader.new_event_log_loader.EventLogDataset'>
<class 'event_log_loader.new_event_log_loader.EventLogDataset'>


### Train Data Insights

In [3]:
# Sepsis Dataset Categories, Features:
sepsis_all_categories = sepsis_train_dataset.all_categories

sepsis_all_categories_cat = sepsis_all_categories[0]
print(sepsis_all_categories_cat)

sepsis_all_categories_num = sepsis_all_categories[1]
print(sepsis_all_categories_num)

for i, cat in enumerate(sepsis_all_categories_cat):
     print(f"sepsis(5) Categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"sepsis (5) Total Amount of Category labels: {cat[1]}")

print('\n')    

for i, num in enumerate(sepsis_all_categories_num):
     print(f"sepsis (5) Numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"sepsis (5) Amount Numerical: {num[1]}")
     
# Get concept_name id:
concept_name = 'concept:name'
concept_name_id = [i for i, cat in enumerate(sepsis_all_categories[0]) if cat[0] == concept_name][0]
print("ID concet name in cat list: ", concept_name_id)

# Output size
concept_name = 'concept:name'
concept_name_size = [cat[1] for _, cat in enumerate(sepsis_all_categories[0]) if cat[0] == concept_name][0]
print("ID concet name in cat list: ", concept_name_size)

# Id of EOS token in activity
eos_value = 'EOS'
eos_id = [v for k, v in sepsis_all_categories[0][concept_name_id][2].items() if k == eos_value][0]
# Get EOS id of concept name list:
print("ID EOS in concept name tensor: ", eos_id)


[('concept:name', 18, {'Admission IC': 1, 'Admission NC': 2, 'CRP': 3, 'EOS': 4, 'ER Registration': 5, 'ER Sepsis Triage': 6, 'ER Triage': 7, 'IV Antibiotics': 8, 'IV Liquid': 9, 'LacticAcid': 10, 'Leucocytes': 11, 'Release A': 12, 'Release B': 13, 'Release C': 14, 'Release D': 15, 'Release E': 16, 'Return ER': 17}), ('org:group', 27, {'?': 1, 'A': 2, 'B': 3, 'C': 4, 'D': 5, 'E': 6, 'EOS': 7, 'F': 8, 'G': 9, 'H': 10, 'I': 11, 'J': 12, 'K': 13, 'L': 14, 'M': 15, 'N': 16, 'O': 17, 'P': 18, 'Q': 19, 'R': 20, 'S': 21, 'T': 22, 'U': 23, 'V': 24, 'W': 25, 'X': 26}), ('lifecycle:transition', 3, {'EOS': 1, 'complete': 2}), ('InfectionSuspected', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('DiagnosticBlood', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('DisfuncOrg', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('SIRSCritTachypnea', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('Hypotensie', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('SIRSCritHeartRate', 5, {'EOS': 1, 'False': 2,

### Input Features for Encoder and Decoder

In [4]:
# Create lists with name of Model features (input)
model_feat_cat = []
model_feat_num = []
for cat in sepsis_all_categories_cat:
    model_feat_cat.append(cat[0])
for num in sepsis_all_categories_num:
    model_feat_num.append(num[0])
model_feat = [model_feat_cat, model_feat_num]
print("Input features encoder: ", model_feat)


Input features encoder:  [['concept:name', 'org:group', 'lifecycle:transition', 'InfectionSuspected', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG'], ['event_elapsed_time', 'day_in_week', 'seconds_in_day', 'Age', 'Leucocytes', 'CRP', 'LacticAcid', 'case_elapsed_time']]


# Model

In [5]:
import sharedCatLSTM.model
importlib.reload(sharedCatLSTM.model)
from sharedCatLSTM.model import SharedCat_LSTM

# Paper hyper-parameters (BPI 2013 row).
hidden_size = 50
num_layers = 1
input_size = 1  # sentinel; SharedCat_LSTM computes the real input size from embeddings.

model = SharedCat_LSTM(
    data_set_categories=sepsis_all_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    model_feat=model_feat,
    input_size=input_size,
    output_size_act=concept_name_size,
)

Data set categories:  ([('concept:name', 18, {'Admission IC': 1, 'Admission NC': 2, 'CRP': 3, 'EOS': 4, 'ER Registration': 5, 'ER Sepsis Triage': 6, 'ER Triage': 7, 'IV Antibiotics': 8, 'IV Liquid': 9, 'LacticAcid': 10, 'Leucocytes': 11, 'Release A': 12, 'Release B': 13, 'Release C': 14, 'Release D': 15, 'Release E': 16, 'Return ER': 17}), ('org:group', 27, {'?': 1, 'A': 2, 'B': 3, 'C': 4, 'D': 5, 'E': 6, 'EOS': 7, 'F': 8, 'G': 9, 'H': 10, 'I': 11, 'J': 12, 'K': 13, 'L': 14, 'M': 15, 'N': 16, 'O': 17, 'P': 18, 'Q': 19, 'R': 20, 'S': 21, 'T': 22, 'U': 23, 'V': 24, 'W': 25, 'X': 26}), ('lifecycle:transition', 3, {'EOS': 1, 'complete': 2}), ('InfectionSuspected', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('DiagnosticBlood', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('DisfuncOrg', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('SIRSCritTachypnea', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('Hypotensie', 5, {'EOS': 1, 'False': 2, 'True': 3, nan: 4}), ('SIRSCritHeartRate', 5,

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


# Training Configuration

In [6]:
import training.camargo_ngram_dataset
import training.train_ngram
importlib.reload(training.camargo_ngram_dataset)
importlib.reload(training.train_ngram)
from training.camargo_ngram_dataset import CamargoNGramDataset
from training.train_ngram import NGramTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

# Paper Table 3: n-gram size 5 for short-trace, simple-SF datasets (Sepsis fits this row).
NGRAM_SIZE = 5

ngram_train = CamargoNGramDataset(sepsis_train_dataset, ngram_size=NGRAM_SIZE,
                                  activity_idx=concept_name_id, eos_idx=eos_id)
ngram_val = CamargoNGramDataset(sepsis_val_dataset, ngram_size=NGRAM_SIZE,
                                activity_idx=concept_name_id, eos_idx=eos_id)
print(f'n-gram train: {len(ngram_train)} samples (from {len(sepsis_train_dataset)} base windows)')
print(f'n-gram val:   {len(ngram_val)} samples (from {len(sepsis_val_dataset)} base windows)')

writer = SummaryWriter(comment="Full_sepsis_camargo_sharedcat_allfeat_ngram5")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# lr=1e-3 (paper-aligned Adam default). The earlier 1e-5 was inherited from the
# U-ED-LSTM setup and is too small for the Camargo baseline.
learning_rate = 1e-3
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4, min_lr=1e-10)

num_epochs = 100
batch_size = 128
shuffle = True

optimize_values = {
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
}

trainer = NGramTraining(
    model=model,
    device=device,
    data_train=ngram_train,
    data_val=ngram_val,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path="../pkl/Sepsis_camargo_sharedcat_allfeat_ngram5.pkl",
)

trainer.train()

n-gram train: 10399 samples (from 11371 base windows)
n-gram val:   2373 samples (from 2665 base windows)
Device:  cpu
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x16660e570>
Epochs: 100  Mini-batch: 128  Shuffle: True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], LR: 0.001
Training:   Avg Loss: 1.9889
Validation: Avg Loss: 1.4711
saving model
Epoch [2/100], LR: 0.001
Training:   Avg Loss: 1.2756
Validation: Avg Loss: 1.2072
saving model
Epoch [3/100], LR: 0.001
Training:   Avg Loss: 1.1016
Validation: Avg Loss: 1.1127
saving model
Epoch [4/100], LR: 0.001
Training:   Avg Loss: 1.0184
Validation: Avg Loss: 1.0995
saving model
Epoch [5/100], LR: 0.001
Training:   Avg Loss: 0.9699
Validation: Avg Loss: 1.0114
saving model
Epoch [6/100], LR: 0.001
Training:   Avg Loss: 0.9379
Validation: Avg Loss: 0.9904
saving model
Epoch [7/100], LR: 0.001
Training:   Avg Loss: 0.9151
Validation: Avg Loss: 1.0167
saving model
Epoch [8/100], LR: 0.001
Training:   Avg Loss: 0.8946
Validation: Avg Loss: 0.9700
saving model
Epoch [9/100], LR: 0.001
Training:   Avg Loss: 0.8759
Validation: Avg Loss: 0.9712
saving model
Epoch [10/100], LR: 0.001
Training:   Avg Loss: 0.8646
Validation: Avg Loss: 1.0320
saving model
Epoch [11/100], LR: 0.001
Training:   A